In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np

In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("CNN_for_TSR.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive/Test/Test

In [ ]:
!pip install wget

In [ ]:
link = "https://d17h27t6h515a5.cloudfront.net/topher/2017/February/5898cd6f_traffic-signs-data/traffic-signs-data.zip"

In [ ]:
import wget

In [ ]:
wget.download(link)

In [ ]:
data = "./"

In [ ]:
!unzip -q traffic-signs-data.zip -d $data

In [ ]:
train_link = data + "train.p"
valid_link = data + "valid.p"
test_link = data + "test.p"

In [ ]:
import pickle

In [ ]:
with open(train_link, mode="rb") as f:
  train = pickle.load(f)

In [ ]:
with open(valid_link, mode="rb") as f:
  valid = pickle.load(f)

In [ ]:
with open(test_link, mode="rb") as f:
  test = pickle.load(f)

In [ ]:
train

In [ ]:
trainX = train["features"]
trainY = train["labels"]

In [ ]:
trainX.shape

In [ ]:
trainX[1].shape

In [ ]:
trainY.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(trainX[50])

In [ ]:
trainY[50]

In [ ]:
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

In [ ]:
classNames[trainY[1]]

In [ ]:
from sklearn.utils import shuffle

In [ ]:
trainX, trainY = shuffle(trainX, trainY)

In [ ]:
plt.imshow(trainX[0])

In [ ]:
classNames[trainY[0]]

In [ ]:
validX = valid["features"]
validY = valid["labels"]


In [ ]:
testX = test["features"]
testY = test["labels"]

In [ ]:
trainX = trainX.astype("float") / 255.0
validX = validX.astype("float") / 255.0
testX = testX.astype("float") / 255.0


In [ ]:
trainX[0]

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()

In [ ]:
trainY = lb.fit_transform(trainY)

In [ ]:
validY = lb.fit_transform(validY)

In [ ]:
valid["labels"][0]

In [ ]:
validY[0]

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential()

In [ ]:
width = 32
height = 32
classes = 43

In [ ]:
shape = (width, height, 3)

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same", input_shape = shape))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(32, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Conv2D(64, (3, 3), padding = "same"))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(512))

In [ ]:
model.add(Activation("relu"))

In [ ]:
model.add(BatchNormalization())

In [ ]:
model.add(Dense(classes))

In [ ]:
model.add(Activation("softmax"))

In [ ]:
model.summary()

In [ ]:
aug = ImageDataGenerator(rotation_range=0.18, zoom_range=0.15, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)

In [ ]:
learning_rate = 0.01

In [ ]:
epochs = 10
batch_size = 64

In [ ]:
opt = SGD(learning_rate=learning_rate, momentum=0.9)

In [ ]:
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
print("Start training")
H = model.fit(aug.flow(trainX, trainY, batch_size=batch_size),
                    validation_data=(validX, validY),
                    steps_per_epoch=trainX.shape[0] // batch_size,
                    epochs=epochs,
                    verbose=1)

In [ ]:
model.save("CNN_for_TSR.h5")

In [ ]:
saved_model = tf.keras.models.load_model("CNN_for_TSR.h5")

In [ ]:
result = saved_model.predict(testX[80:81])

In [ ]:
result

In [ ]:
import numpy as np



In [ ]:
final = np.argmax(result)

In [ ]:
final = classNames[final]

In [ ]:
final

In [ ]:
plt.imshow(test["features"][80])

In [ ]:
from tensorflow.keras.models import load_model

import cv2
import numpy as np
# Tải mô hình
model = load_model("traffic_classifier.h5")
print("Mô hình đã được tải thành công!")


def preprocess_image(image_path):
    # Đọc ảnh từ đường dẫn
    image = cv2.imread(image_path)
    # Resize ảnh về kích thước 32x32 (hoặc kích thước phù hợp với mô hình)
    image = cv2.resize(image, (32, 32)) # Changed to 32x32 to match the model's input shape
    # Chuẩn hóa giá trị pixel về [0, 1]
    image = image / 255.0
    # Thêm chiều batch
    image = np.expand_dims(image, axis=0)
    return image

# Đường dẫn ảnh thử nghiệm
image_path = "02336.png"

# Xử lý ảnh
processed_image = preprocess_image(image_path)

# Dự đoán với mô hình
predictions = model.predict(processed_image)
predicted_class = np.argmax(predictions)  # Lấy nhãn có xác suất cao nhất
# print("Dự đoán kết quả:")
# final = np.argmax(predictions)
# final = classNames[final]
# print(final)

# Mapping nhãn số sang tên biển báo (ví dụ)
# Define classNames here as it was not found in the current scope
classNames = {0: 'Speed limit (20km/h)',
 1: 'Speed limit (30km/h)',
 2: 'Speed limit (50km/h)',
 3: 'Speed limit (60km/h)',
 4: 'Speed limit (70km/h)',
 5: 'Speed limit (80km/h)',
 6: 'End of speed limit (80km/h)',
 7: 'Speed limit (100km/h)',
 8: 'Speed limit (120km/h)',
 9: 'No passing',
 10: 'No passing for vehicles over 3.5 metric tons',
 11: 'Right-of-way at the next intersection',
 12: 'Priority road',
 13: 'Yield',
 14: 'Stop',
 15: 'No vehicles',
 16: 'Vehicles over 3.5 metric tons prohibited',
 17: 'No entry',
 18: 'General caution',
 19: 'Dangerous curve to the left',
 20: 'Dangerous curve to the right',
 21: 'Double curve',
 22: 'Bumpy road',
 23: 'Slippery road',
 24: 'Road narrows on the right',
 25: 'Road work',
 26: 'Traffic signals',
 27: 'Pedestrians',
 28: 'Children crossing',
 29: 'Bicycles crossing',
 30: 'Beware of ice/snow',
 31: 'Wild animals crossing',
 32: 'End of all speed and passing limits',
 33: 'Turn right ahead',
 34: 'Turn left ahead',
 35: 'Ahead only',
 36: 'Go straight or right',
 37: 'Go straight or left',
 38: 'Keep right',
 39: 'Keep left',
 40: 'Roundabout mandatory',
 41: 'End of no passing',
 42: 'End of no passing by vehicles over 3.5 metric tons'}

# Use the original classNames dictionary to create the class_names list
class_names = [classNames[i] for i in range(len(classNames))]

print(f"Biển báo được nhận diện: {class_names[predicted_class]}")